# feature engineering 2.0
- following from second meeting with Greenberg 
- feedback included:
-	CARD might be better because we know they have diagnosis 
-	Rerun and make a more specific and comprehensive list of features and ensure that you know what each feature is. 
o	When rerunning if you want to include individual items do it but not domains (not sure what this meant but figure it out and try it)
o	Surprised AQ and Sex not big predictors 
-	Rerun but target group excludes people ‘with diagnosis’ but score below threshold of AQ
-	Rerun with AQ as target Var e.g., 0 class below a 6? and target case 6 and above?
-	Rerun but with my own PCA on each of the questions 
-	Rerun with AQ cut off as target var 
-	Also ‘other’ could have been a text option so maybe look into this?
-	Oh and email don’t use linked in lol 


# notebook uses beginning of feature_engineering 1.0 to set up the same dataset



- loading matched balanced data

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.feature_selection import VarianceThreshold
# Add these imports to your notebook
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier

# load matched data 
df = pd.read_csv('../data/processed/data_c4_matched_balanced.csv')

# 1. feature creation 

# age group bins
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], labels=['0-18', '19-30', '31-45', '46-60', '61+'])

#non linear transformation 
df['log_aq_total'] = np.log1p(df['aq_total'])
df['sqrt_age'] = np.sqrt(df['age'])

# interaction terms
df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
df['sqp_aq_interaction'] = df['spq_total'] * df['aq_total']
df['age_x_eq'] = df['age'] * df['eq_total']

# questionnaire score ratios 
df['aq_spq_ratio'] = df['aq_total'] / (df['spq_total'] + 1e-8)
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

#boolean: high aq (above 1 std)
df['high_aq'] = (df['aq_total'] > df['aq_total'].mean() + df['aq_total'].std()).astype(int)

# 2. feature reduction/selection

# remove highly correlated features 
# Only use numeric columns for correlation
numeric_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df = df.drop(columns=to_drop)

# drop low variance features 
# Only apply VarianceThreshold to numeric columns
feature_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
selector = VarianceThreshold(threshold=0.1)
selector.fit(df[feature_cols])
low_variance_cols = feature_cols[~selector.get_support()]
df = df.drop(columns=low_variance_cols)

# 3. one-hot encode new categorical features 
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)

# 4. save engineered dataset 
df.to_csv('../data/processed/data_c4_balanced_fe.csv', index=False)

print("feature engineering complete. new shape:", df.shape)
print("columns:", df.columns.tolist())

# training baseline models
- Log reg
- RF
- XGB

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

# df load data
df = pd.read_csv('../data/processed/data_c4_balanced_fe.csv')
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)

# logistic reg
logreg = LogisticRegression(max_iter=2000, class_weight='balanced')
logreg.fit(x_train, y_train)
print("logistic regression:")
print(classification_report(y_val, logreg.predict(x_val)))
print("ROC-AUC:", roc_auc_score(y_val, logreg.predict_proba(x_val)[:, 1]))

#random forest 
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced')
rf.fit(x_train, y_train)
print("random forest:")
print(classification_report(y_val, rf.predict(x_val)))
print("ROC-AUC:", roc_auc_score(y_val, rf.predict_proba(x_val)[:, 1]))

# xgboost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(x_train, y_train)
print("xgboost:")
print(classification_report(y_val, xgb.predict(x_val)))
print("ROC-AUC:", roc_auc_score(y_val, xgb.predict_proba(x_val)[:, 1]))


# threshold moving

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report, roc_auc_score

# Random Forest threshold moving
rf_probs = rf.predict_proba(x_val)[:, 1]

rf_prec, rf_rec, rf_thresholds = precision_recall_curve(y_val, rf_probs)
rf_f1s = 2 * (rf_prec * rf_rec) / (rf_prec + rf_rec + 1e-8)
rf_best_thresh = rf_thresholds[np.argmax(rf_f1s)]
print(f"Random Forest - Best threshold for F1: {rf_best_thresh:.3f}")

# evaluate RF at the best threshold 
rf_pred_thresh = (rf_probs >= rf_best_thresh).astype(int)
print("Random Forest validation set performance at best threshold:")
print(classification_report(y_val, rf_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val, rf_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val, rf_probs):.3f}")

# XGBoost threshold moving
xgb_probs = xgb.predict_proba(x_val)[:, 1]

xgb_prec, xgb_rec, xgb_thresholds = precision_recall_curve(y_val, xgb_probs)
xgb_f1s = 2 * (xgb_prec * xgb_rec) / (xgb_prec + xgb_rec + 1e-8)
xgb_best_thresh = xgb_thresholds[np.argmax(xgb_f1s)]
print(f"\nXGBoost - Best threshold for F1: {xgb_best_thresh:.3f}")

# evaluate XGBoost at the best threshold 
xgb_pred_thresh = (xgb_probs >= xgb_best_thresh).astype(int)
print("XGBoost validation set performance at best threshold:")
print(classification_report(y_val, xgb_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val, xgb_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val, xgb_probs):.3f}")

# feature importance

In [ ]:
import pandas as pd

# for random forest
importances = pd.Series(rf.feature_importances_, index=x_train.columns)
print("Top 20 random forest features:")
print(importances.sort_values(ascending=False).head(20))

# for xgboost
importances_xgb = pd.Series(xgb.feature_importances_, index=x_train.columns)
print("Top 20 xgboost features:")
print(importances_xgb.sort_values(ascending=False).head(20))

# shap for interpretable analysis
import shap
import numpy as np

# for RF - using a subsample for faster SHAP computation
# Take a smaller sample (e.g., 20% of validation data) for efficiency
sample_size = min(500, int(0.2 * len(x_val)))
x_val_sample = x_val.sample(n=sample_size, random_state=42)

# Create explainer and compute SHAP values only on the subsample
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(x_val_sample)

# Generate plots with the subsampled data
shap.summary_plot(shap_values, x_val_sample, plot_type="bar", max_display=20)
shap.summary_plot(shap_values, x_val_sample, max_display=20)